# Chain-of-Thought and Thinking Models

> 357 × 289 — even a human needs to work through it step by step. A standard LLM tries to output the answer directly, and one-shot accuracy is naturally low.
>
> Chain-of-Thought (CoT) has the model write out intermediate steps before giving the answer, boosting accuracy significantly. Thinking models like o1 and DeepSeek-R1 go further: they do not just add thinking at inference time; they learn to "think before answering" during training. This section breaks down the principles behind this.

The core idea of CoT can be stated in one sentence: add "Let's think step by step" to the prompt, and the model will write out its reasoning process before giving the final answer.

Writing out intermediate steps helps because each step reduces the problem size, and if an intermediate step goes wrong, later steps still have a chance to correct it. But this is only a prompt-level trick — the model itself has not learned to "think before answering."

Thinking models go further — they are shaped by reinforcement learning during training to produce an internal chain of thought before outputting the final answer. The key finding of DeepSeek-R1 is: you do not need human-annotated thinking chains. Using only the outcome reward signal of "is the answer correct," the model can spontaneously learn to think before answering.


## 1. The Reasoning Gap in LLMs

```
User: 357 × 289 = ?

Inside a standard LLM:
  input tokens → embedding → transformer × N → output "103173"
  
  It didn't "compute"! It just "guessed" a number based on patterns in training data.
  If the specific combination 357×289 never appeared in training, it usually guesses wrong.
```

**Root cause**: Every layer of a Transformer has a fixed amount of computation. Whether the question is "1+1" or "calculus," the compute budget is the same — it has no built-in capacity for "multi-step reasoning."

Just like asking you to mentally calculate 357×289 — you cannot give the answer directly. You need:
1. 357×200 = 71400
2. 357×80 = 28560
3. 357×9 = 3213
4. 71400 + 28560 + 3213 = 103173

An LLM also needs this "step-by-step computation" process — that is exactly what CoT provides.


## 2. Chain-of-Thought (CoT)

The core idea of CoT is extremely simple: **demonstrate "write the reasoning process first, then give the answer" in the prompt**.

```
Prompt without CoT:
  Q: 357 × 289 = ?
  A: 103173  ← model guesses directly

Prompt with CoT (Few-shot):
  Q: 123 × 45 = ?
  A: 123×40=4920, 123×5=615, 4920+615=5535. The answer is 5535.
  
  Q: 357 × 289 = ?
  A:  ← model imitates the format above, writing the process first then the answer
```

**Why does it work?** Because when the model generates the reasoning process, the result of each intermediate step becomes "context" for subsequent steps. The Transformer's attention can see the intermediate results computed earlier, allowing it to reason based on them.

In essence: **CoT turns "a problem one forward pass cannot solve" into "multiple forward passes working in relay."**


In [ ]:
# Live computation demo: why CoT works
import numpy as np
import random

print("=== Why CoT Works ===")
print()

target = 357 * 289

# Without CoT: model must compute in one step
print("Without CoT:")
print(f"  Model must compute 357 × 289 in one step")
print(f"  Correct answer: {target:,}")
np.random.seed(42)
guesses = np.random.randint(80000, 120000, size=5)
closest = guesses[np.argmin(np.abs(guesses - target))]
print(f"  Model can only 'guess': {closest:,} (off by {abs(closest - target):,})")
print(f"  → One shot is too hard!")
print()

# With CoT: step-by-step
print("With CoT (step by step):")
result = 0
steps = []
for multiplier, label in [(200, "200"), (80, "80"), (9, "9")]:
    partial = 357 * multiplier
    result += partial
    steps.append(partial)
    print(f"  357 × {label} = {partial:,}")

total = sum(steps)
print(f"  {' + '.join(f'{s:,}' for s in steps)} = {total:,}")
print()
print(f"✅ CoT Result: {total:,} == Correct answer {target:,}")
print()
print("Key insight: each step's result can be referenced by subsequent steps.")
print("This trades 'token generation time' for 'computational depth.'")


### 2.5 Self-Consistency: Ask the Same Question Multiple Times, Vote on the Answer

CoT has one problem: **the model might make an error on a particular reasoning chain**. Ask the model the same math question 5 times (with temperature > 0), and it may produce 3 different reasoning processes and answers.

**The core idea of Self-Consistency**:
```
Same question → sample N different CoT reasoning chains → vote on the final answer → the answer with the most votes wins
```

**Why does it work?**

Intuition: you take a hard problem and ask 5 classmates —
- Each classmate might go wrong at some step
- But different classmates are **likely to make different errors, unlikely to make the same error**
- The correct answer is unique, and it appears most frequently across multiple correct or partially correct chains

Mathematically: if a single chain's accuracy is p, the probability that the majority of N chains are correct increases with N:
```
p=0.6, N=1: accuracy 60%
p=0.6, N=5: probability of at least 3 correct = C(5,3)p^3(1-p)^2 + ... ≈ 68%
p=0.7, N=5: probability of at least 3 correct ≈ 84%  ← significant improvement!
```

**What you do NOT need**: no retraining, no architecture changes — just **multiple sampling + voting** at inference time.

The following simulation demonstrates the full process:


In [ ]:
# ============================================================
# Self-Consistency demo: same question, 5 reasoning chains, vote on the answer
# ============================================================
import random
random.seed(42)

print("=" * 70)
print("Problem: Xiaoming has 15 apples. He gives Xiaohong 3, buys 8 more,")
print("         then eats 2, and finally gives half of the remainder to Xiaogang.")
print("         How many apples does Xiaoming have now?")
print("=" * 70)

# Simulate 5 different reasoning chains (mimicking model sampling at temperature > 0)
# Each chain shows reasoning process + final answer

reasoning_chains = [
    {
        "id": 1,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 2,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 3,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: gives half to Xiaogang → remainder 18 - 9 = 9",  # reasoning correct, answer correct
        ],
        "answer": 9,
        "correct": True
    },
    {
        "id": 4,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3 → 15 - 3 = 12",
            "Step 3: buys 8 more → 12 + 8 = 20",
            "Step 4: eats 2 → 20 - 2 = 18",
            "Step 5: forgot to divide by 2! → answer 18",  # ← forgot the last step!
        ],
        "answer": 18,
        "correct": False
    },
    {
        "id": 5,
        "reasoning": [
            "Step 1: starts with 15",
            "Step 2: gives Xiaohong 3, buys 8 → total 15 - 3 + 8 = 20",
            "Step 3: eats 2 -> 20 - 2 = 18",
            "Step 4: gives half to Xiaogang → gives 18/2 = 9, keeps 9",
        ],
        "answer": 9,
        "correct": True
    },
]

# Print each reasoning chain
for chain in reasoning_chains:
    print(f"\n--- Chain #{chain['id']} ---")
    for step in chain['reasoning']:
        print(f"  {step}")
    status = "correct" if chain['correct'] else "wrong"
    print(f"  Answer: {chain['answer']} apples  {status}")

# ============================================================
# Majority voting
# ============================================================
print(f"\n{'=' * 70}")
print("Voting")
print(f"{'=' * 70}")

from collections import Counter
answers = [c['answer'] for c in reasoning_chains]
vote_counts = Counter(answers)

for ans, count in vote_counts.most_common():
    correct_mark = "correct" if ans == 9 else "wrong"
    bar = "#" * count
    print(f"  Answer {ans}: {count} votes {bar} ({correct_mark})")

winner = vote_counts.most_common(1)[0][0]
winner_is_correct = (winner == 9)

print(f"\n  Final answer (majority vote): {winner} apples")
print(f"     Correct: {'✓ correct！' if winner_is_correct else '✗ wrong'}")

# ============================================================
# Comparison: single vs Self-Consistency
# ============================================================
print(f"\n{'=' * 70}")
print("Comparison")
print(f"{'=' * 70}")
print(f"  Single sampling (pick one at random): accuracy = 4/5 = 80%")
print("  Self-Consistency (5 chains vote): accuracy = 100% (all correct this time!)")
print(f"")
print(f"  Key insight:")
print(f"  - Chain #4 made an error at the last step (forgot to divide by 2)")
print("  - But the other 4 chains were correct, so the vote result = 9 (correct)")
print(f"  - Self-Consistency swallowed that error!")
print(f"")
print(f"  This is the power of Self-Consistency:")
print("  Majority rules -- one chain making an error is fine; multiple chains will not make the same mistake.")


#### 2.5.1 Self-Consistency vs Thinking Models

Self-Consistency and the Thinking models (o1, R1) discussed later are fundamentally different:

| | Self-Consistency | Thinking Models |
|------|---------|-------|
| **How** | Multiple sampling + voting at inference time | Learns to "self-reflect" during training |
| **Cost** | N× inference cost (sample N times) | 1 inference, but internal thinking is long |
| **Training needed** | No | Yes (RL training) |
| **Key capability** | Diverse sampling | Self-verification, error correction, backtracking |

**In essence**:
- Self-Consistency is **external** error correction — relying on multiple samples to "hit" the correct answer
- Thinking models are **internal** error correction — checking and correcting within a single reasoning chain

The latter is harder to train, but more efficient at inference (no repeated sampling needed).
R1's self-verification capability can be understood as a **built-in, automatic Self-Consistency** — it performs "multi-angle checks" in its head.


## 3. From CoT to Thinking Models

The problem with CoT is that the thinking process is exposed to the user. Users don't necessarily want to see your scratch paper.

Thinking models separate the "scratch paper" from the "final answer":

```text
What the user sees:
  Q: 357 × 289 = ?
  A: 103173

What the model actually generates internally:
  <think>
  357×200=71400
  357×80=28560
  357×9=3213
  71400+28560+3213=103173
  </think>
  103173
```

The `<think>` and `</think>` here are thinking markers.
They act like a pair of brackets: the thinking draft goes in between, and the final answer follows.

The frontend or API can choose to display only the answer after `</think>`, collapsing or hiding the intermediate draft.

**In essence**: Thinking Models = engineered packaging of CoT. The thinking process is still there, just managed by special markers in separate zones.


### 3.5 How Think Markers Work Under the Hood

#### 3.5.1 Why `<think>` is not ordinary text

You might think: isn't `<think>` just a few characters? The model just outputs these characters and that's it?

It's not that simple.

If `<think>` were just ordinary text, three problems could arise:

1. **Easily fragmented by the tokenizer**: `<think>` might become three tokens: `<`, `think`, `>`.
2. **Unstable end boundary**: The model might accidentally write `</think>` inside the draft, causing the frontend to end thinking prematurely.
3. **Hard to control during training**: It's difficult to stably distinguish between the thinking zone and the answer zone.

So the more robust approach is to add `<think>` and `</think>` as **special tokens**.

That is, they should have their own independent IDs, just like `<BOS>`, `<EOS>`:

```text
<think>   -> 100
</think>  -> 101
```

This way, when the model generates ID 100, it enters the thinking zone; when it generates ID 101, it exits the thinking zone.


#### 3.5.2 Current Practice: How to Add New Thinking Markers to Training

In modern training, adding new markers like `<think>` generally falls into two scenarios.

**Scenario A: Training tokenizer and model from scratch**

Add these markers to the tokenizer's special token list from the start:

```text
<BOS>, <EOS>, <PAD>, <think>, </think>
```

This way, during tokenizer training they won't be fragmented; during model training, they'll learn embeddings just like regular tokens.

**Scenario B: Continuing training on an existing model**

This is the more common path. For example, if you take an open-source base model and want it to learn a new thinking format, you typically do 4 things:

1. **Add markers to the tokenizer**: Add `<think>` and `</think>` as special tokens.
2. **Expand model embeddings**: The tokenizer vocabulary grows, so the model's `Embedding` and output layers must also grow.
3. **Prepare formatted data**: Training samples must actually contain `<think>...</think>`.
4. **Continue SFT / RL**: Let the model learn via loss or reward when to open and close thinking.

Common pseudocode in practice:

```python
new_tokens = {"additional_special_tokens": ["<think>", "</think>"]}
num_added = tokenizer.add_special_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))

# Then continue training with data containing <think>...</think>
```

One thing that's easy to misunderstand: **adding tokens just gives the model a new pen; it doesn't mean it can write reasoning.**
What actually teaches the model to think is the training data, loss mask, and reward design that follow.


In [ ]:
# Minimal example: training sample after adding <think>  markers
vocab = {
    "<BOS>": 0,
    "<EOS>": 1,
    "<PAD>": 2,
    "user": 3,
    "assistant": 4,
    "answer": 5,
    "357": 6,
    "289": 7,
    "103173": 8,
}

new_symbols = ["<think>", "</think>"]
for symbol in new_symbols:
    if symbol not in vocab:
        vocab[symbol] = len(vocab)

train_tokens = [
    "<BOS>",
    "user", "357", "289",
    "assistant", "<think>", "357", "289", "103173", "</think>",
    "answer", "103173",
    "<EOS>",
]
train_ids = [vocab[token] for token in train_tokens]

print("New symbols:")
for symbol in new_symbols:
    print(f"  {symbol} -> ID {vocab[symbol]}")

print()
print("Training sample:")
print(train_tokens)
print()
print("Training IDs:")
print(train_ids)
print()
print("Key observation: the model doesn't understand '<think>' the English word，")
print("but learns through training that between ID 9 and ID 10 it should write a reasoning draft.")

In [ ]:
# ============================================================
# Demo: How Special Tokens are tokenized
# ============================================================
print("=== Special Token vs Plain Text: Tokenization Comparison ===\n")

# Simulate tokenizer behavior -- simplified BPE-style tokenizer
# Core question: what happens if <think> is just plain text?

# Assume this is the tokenizer vocabulary (simplified)
vocab = {
    "I": 1, "like": 2, "eat": 3, "apple": 4, "orange": 5,
    "think": 7, "<": 9, ">": 10, "/": 11,
    "reasoning": 13, "process": 14, "answer": 15, "is": 16,
    # Key: if set as special token, it gets a unique ID
    # "<think>": 100,
    # "</think>": 101,
}

print("Vocabulary (plain tokens, no special tokens):")
for k, v in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  '{k}' -> {v}")
print()

# ========================================
# Scenario 1: without special tokens
# ========================================
print("=" * 55)
print("Scenario 1: <think> is just plain text (not in vocabulary)")
print("=" * 55)

def tokenize_plain(text, vocab):
    """Simulate BPE/word-level tokenizer without special tokens"""
    tokens = []
    i = 0
    while i < len(text):
        # Longest match for regular tokens
        matched = None
        for word in sorted(vocab.keys(), key=len, reverse=True):
            if text[i:].startswith(word):
                matched = (word, vocab[word])
                break
        if matched:
            tokens.append(matched)
            i += len(matched[0])
        else:
            # Character-level fallback
            ch = text[i]
            tid = ord(ch) % 50 + 20  # Simulate unknown token id
            tokens.append((ch if ch != ' ' else '_', tid))
            i += 1
    return tokens

text = "Ilike<think>reasoningprocess</think>answerisapple"
tokens_plain = tokenize_plain(text, vocab)

print(f"Input: {text}")
print(f"\nTokenization result ({len(tokens_plain)} tokens):")
ids = []
for word, tid in tokens_plain:
    ids.append(tid)
    flag = " [fragmented!]" if tid in [9,10,11] else ""
    print(f"  [{word:12s}] -> ID {tid:3d}{flag}")

print(f"\nToken ID sequence: {ids}")
print(f"\nKey problems:")
print(f"  <think> is split into 3 tokens: < think > (3 separate IDs)")
print(f"  If thinking content contains '<' (e.g. x<5) -> chaos")
print(f"  Cannot precisely locate thinking boundaries via token ID")
print(f"  The model must learn to output exactly these 5 tokens in order -> extremely hard")

# ========================================
# Scenario 2: with special tokens added
# ========================================
print(f"\n{'='*55}")
print("Scenario 2: <think> as Special Token (correct approach)")
print("=" * 55)

# Add special tokens to vocabulary
vocab_with_special = vocab.copy()
vocab_with_special["<think>"] = 100
vocab_with_special["</think>"] = 101

print("Added to vocabulary:")
print("  '<think>'  -> 100  (special token)")
print("  '</think>' -> 101  (special token)")
print()

def tokenize_with_special(text, vocab):
    """
    Tokenizer scans for special tokens first (longest match priority),
    then regular tokens. This is how all real tokenizers work.
    """
    tokens = []
    i = 0
    while i < len(text):
        # Match special tokens first (ID >= 100)
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: (-len(x), x)):
            if text[i:].startswith(word):
                matched = (word, vocab[word])
                break
        if matched:
            tokens.append(matched)
            i += len(matched[0])
        else:
            ch = text[i]
            tokens.append((ch, ord(ch) % 50 + 20))
            i += 1
    return tokens

tokens_special = tokenize_with_special(text, vocab_with_special)

print(f"Input: {text}")
print(f"\nTokenization result ({len(tokens_special)} tokens):")
ids_special = []
for word, tid in tokens_special:
    ids_special.append(tid)
    flag = " [special]" if tid >= 100 else ""
    print(f"  [{word:12s}] -> ID {tid:3d}{flag}")

print(f"\nToken ID sequence: {ids_special}")
print(f"Token count: {len(tokens_plain)} -> {len(tokens_special)}")
print(f"\nKey advantages:")
print(f"  <think> = one token (ID=100), not fragmented")
print(f"  '<' or '>' inside thinking content have no effect (special matched first)")
print(f"  Check ID==100/101 to precisely locate thinking boundaries")
print(f"  Model only needs to learn to output token 100 at the right position")

# ========================================
# Demo with real tokenizer (if tiktoken is available)
# ========================================
print(f"\n{'='*55}")
print("Real tokenizer approach")
print("=" * 55)

try:
    import tiktoken

    # GPT-2 tokenizer (no think special token)
    enc = tiktoken.get_encoding("gpt2")
    plain_result = enc.encode("<think>")
    print(f"\nGPT-2 tokenizer encoding '<think>':")
    print(f"  Token IDs: {plain_result}")
    print(f"  Each token: {[enc.decode([t]) for t in plain_result]}")
    print(f"  -> Split into {len(plain_result)} tokens (because '<think>' is not in GPT-2 vocabulary)")

    # Explain the real approach
    print(f"\nDeepSeek-R1 / Qwen3 approach:")
    print(f"  1. Add special tokens to the tokenizer vocabulary:")
    print(f"     tokenizer.add_special_tokens({{'<think>': ..., '</think>': ...}})")
    print(f"  2. Expand model embedding matrix (add a row for the new token)")
    print(f"  3. Now '<think>' is a single complete token")

except ImportError:
    print("\n(tiktoken not installed, skipping real tokenizer demo)")
    print("pip install tiktoken to run the demo above")
    print()
    print("Using GPT-4 cl100k_base as an example:")
    print("  '<think>' if not in vocabulary -> may be split into multiple BPE tokens")
    print("  After adding to vocabulary -> 1 token -> model learning cost reduced 5x")

print(f"\nSummary: add_special_tokens() is the first step in training a thinking model.")


### 3.6 How to Compute Loss When Training Thinking Models

With special tokens like `<think>` and `</think>`, an important question arises:

**During training, should thinking tokens contribute to the loss?**

There are three strategies, each with tradeoffs. Let us first look at a concrete example:

```
Complete assistant output token sequence:
[<think>, 3, 1, 2, x, 2, 0, 0, =, 7, 1, 4, 0, 0, </think>, 1, 0, 3, 1, 7, 3]
  |<------------ thinking tokens ----------->|<--- answer tokens --->|
                  14 tokens                           6 tokens
```

#### Strategy Comparison

| | Strategy A: Full Loss | Strategy B: Answer Only | Strategy C: Selective Weighting |
|---|---|---|---|
| Thinking token loss | Included | Excluded | Included but down-weighted (x0.1) |
| Answer token loss | Included | Included | Included |
| Special token loss | Included | Excluded | Excluded |
| Advantage | Simple and direct; thinking quality is also optimized | Only focuses on final answer; thinking has high freedom | Balances both; thinking is guided but does not dominate |
| Disadvantage | Thinking may become performance rather than genuine reasoning | Thinking process may degrade or become incoherent | Requires tuning the weight hyperparameter |
| Representative model | DeepSeek-R1 (full loss in RL phase) | Early thinking experiments | Some RL experiments |

#### Why does DeepSeek-R1 choose full loss?

The key insight from the R1 paper: **if thinking tokens do not contribute to loss, the model has no incentive to think.**

During RL training, the model discovers that "less thinking -> faster answer output -> earlier reward"
causes thinking to become shorter and shorter, eventually degrading.

Full loss means: the model must take every thinking token seriously, because they all contribute loss signals.

But full loss has a cost too: the model may learn "performative thinking" --
generating tokens that look like reasoning but are actually useless, simply because those tokens have low loss (the model is confident about them).
Fortunately, the RL reward signal (only correct answers get points) can counteract this problem.

#### How to implement it?

The core is constructing a **loss_mask** -- a 0/1 array with the same shape as labels:

```python
# labels:   [100, 3, 1, 2, 11, ..., 101, 1, 0, 3, 1, 7, 3]
# loss_mask:[  0, 1, 1, 1,  1, ...,   0, 1, 1, 1, 1, 1, 1]
#              ^ special token excluded       ^ special token excluded
#
# final loss = cross_entropy(logits, labels) * loss_mask
# -> special token positions have loss=0 -> no gradient
```

Below is a complete code implementation of all three strategies.

In [ ]:
# ============================================================
# Complete implementation: loss_mask construction for three strategies
# ============================================================
import torch
import torch.nn.functional as F

import torch.nn as nn

print("=== Thinking Model Loss Mask Complete Implementation ===\n")

# Simulate a batch of token sequences
# Special token IDs: <think>=100, </think>=101
# Regular tokens: 1-50
# PAD=0, IGNORE=-100 (PyTorch standard ignore value)
THINK_START = 100
THINK_END = 101
IGNORE = -100

# Simulated training data: labels for two samples
# Sample 1: <think> 3,1,2,11,2,0,0 </think> 1,0,3,1,7,3  <- normal thinking->answer
# Sample 2: <think> 5,x,6,=,3,0 </think> 3,0              <- brief thinking->answer
batch_labels = torch.tensor([
    [100,   3,   1,   2,  11,   2,   0,   0, 101,   1,   0,   3,   1,   7,   3,   0],
    [100,   5,  12,   6,  13,   3,   0, 101,   3,   0,   0,   0,   0,   0,   0,   0],
])  # shape: [2, 16]

# Simulated model output logits
VOCAB_SIZE = 200
logits = torch.randn(2, 16, VOCAB_SIZE)

print("Batch labels:")
print(batch_labels)
print()

# ============================================================
# Strategy A: Full Loss -- all tokens contribute to loss
# ============================================================
print("=" * 60)
print("Strategy A: Full Loss (all count)")
print("=" * 60)

def make_full_loss_mask(labels, ignore_id=0):
    """Simplest mask: only ignore PAD tokens"""
    return (labels != ignore_id).float()

mask_full = make_full_loss_mask(batch_labels)
print("loss_mask (0=PAD excluded, 1=included):")
print(mask_full.int())
print()

# Compute loss
loss_full = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    batch_labels.view(-1),
    ignore_index=0,  # PAD excluded
    reduction='none'
).view(2, 16)

print("Per-position loss:")
print(loss_full)
print()

total_full = loss_full.sum() / mask_full.sum()
print(f"Average loss (full): {total_full:.4f}")
print("-> Every token in thinking and answer is optimized")
print()

# ============================================================
# Strategy B: Answer Only -- only tokens after </think>
# ============================================================
print("=" * 60)
print("Strategy B: Answer Only (only answer section)")
print("=" * 60)

def make_answer_only_mask(labels, think_start=100, think_end=101, ignore_id=0):
    """
    Only let tokens after </think> contribute to loss.
    Logic: find the </think> position in each sequence, tokens after it count.
    """
    mask = torch.zeros_like(labels, dtype=torch.float)

    for b in range(labels.shape[0]):
        # Find </think> position
        end_positions = (labels[b] == think_end).nonzero(as_tuple=True)[0]
        if len(end_positions) > 0:
            end_pos = end_positions[0].item()
            # After </think> (not including it) to before EOS/PAD
            for pos in range(end_pos + 1, labels.shape[1]):
                if labels[b, pos] == ignore_id:
                    break
                mask[b, pos] = 1.0
        # If no think_end found -> entire sequence may be non-thinking mode -> count all

    return mask

mask_answer_only = make_answer_only_mask(batch_labels)
print("loss_mask (0=excluded, 1=included):")
print(mask_answer_only.int())
print()

# Label which tokens belong to thinking, which to answer
print("Token labels (T=thinking, A=answer, S=special, P=PAD):")
for b in range(2):
    row = ""
    for pos in range(16):
        tid = batch_labels[b, pos].item()
        if tid == 0:
            row += " P "
        elif tid == 100:
            row += "[S "
        elif tid == 101:
            row += " S]"
        elif mask_answer_only[b, pos] == 1:
            row += " A "
        else:
            row += " T "
    print(f"  Sample{b}: {row}")
print()

# Construct masked labels: positions not in loss set to IGNORE
labels_masked_B = batch_labels.clone()
labels_masked_B[mask_answer_only == 0] = IGNORE

loss_answer_only = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    labels_masked_B.view(-1),
    ignore_index=IGNORE,
)
print(f"Average loss (answer only): {loss_answer_only:.4f}")
print("-> Only answer tokens are optimized, thinking is free-form")
print()

# ============================================================
# Strategy C: Selective Weighting -- thinking down-weighted
# ============================================================
print("=" * 60)
print("Strategy C: Selective Weighting (thinking x0.1)")
print("=" * 60)

def make_selective_weight_mask(labels, think_start=100, think_end=101,
                                thinking_weight=0.1, ignore_id=0):
    """
    Both thinking and answer tokens contribute to loss, but with different weights:
    - thinking token: weight = 0.1
    - answer token:   weight = 1.0
    - special token:  excluded
    """
    weight_mask = torch.zeros_like(labels, dtype=torch.float)

    for b in range(labels.shape[0]):
        in_thinking = False
        for pos in range(labels.shape[1]):
            tid = labels[b, pos].item()

            if tid == think_start:
                in_thinking = True
                continue  # special token itself excluded
            elif tid == think_end:
                in_thinking = False
                continue  # special token itself excluded
            elif tid == ignore_id:
                break  # PAD excluded

            if in_thinking:
                weight_mask[b, pos] = thinking_weight  # thinking: 0.1
            else:
                weight_mask[b, pos] = 1.0               # answer: 1.0

    return weight_mask

weight_mask = make_selective_weight_mask(batch_labels)
print("Weight matrix:")
for b in range(2):
    print(f"  Sample{b}: {[f'{w:.1f}' for w in weight_mask[b].tolist()]}")
print()

# Weighted loss
loss_per_token = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    batch_labels.view(-1),
    ignore_index=0,
    reduction='none'
).view(2, 16)

weighted_loss = (loss_per_token * weight_mask).sum() / weight_mask.sum()
print(f"Average loss (selective weighting): {weighted_loss:.4f}")
print("-> thinking gets 10% optimization signal, answer gets 100%")

# ============================================================
# Three-strategy comparison
# ============================================================
print(f"\n{'='*60}")
print("Three Strategy Comparison Summary")
print("=" * 60)
print()
print(f"  | Metric                  | Full Loss | Answer Only | Selective |")
print(f"  |-------------------------|-----------|-------------|----------|")
print(f"  | Avg Loss                | {total_full:.4f}   | {loss_answer_only:.4f}     | {weighted_loss:.4f}  |")
print(f"  | Thinking optimized      | Yes       | No          | 0.1x     |")
print(f"  | Answer optimized        | Yes       | Yes         | Yes      |")
print(f"  | Thinking won't degrade  | Yes       | No          | Yes      |")
print(f"  | Implementation          | Low       | Medium      | Medium   |")
print()

print("How to choose in real training?")
print("  SFT phase -> commonly use Answer Only (let the model learn the format first)")
print("  RL phase -> commonly use Full Loss (DeepSeek-R1 approach)")
print("  Transition -> use Selective Weighting for smooth transition")
print()
print("Common to all strategies: special tokens (<think>/</think>) do not count in loss.")
print("Because these tokens are just markers, their prediction has no meaning.")

## 4. How to Train a Thinking Model

Training a thinking model has four steps:

```
Step 1: Cold-start SFT
  Collect a few thousand high-quality examples with thinking process
  Format: Q -> thinking... -> A
  Use these data for supervised fine-tuning, teaching the model the "think before answering" format

Step 2: RL reasoning training (the core!)
  Use reinforcement learning to train the model's reasoning ability
  Reward signals:
    - correct answer -> +1
    - wrong answer -> -1
    - correct format (has thinking tags) -> +0.1
    - language consistency (thinking and answer in same language) -> +0.1
  The model explores better reasoning paths on its own

Step 3: Rejection sampling + SFT
  Use the trained model to generate lots of question->thinking->answer data
  Only keep samples with correct answers
  Use this high-quality data for another round of SFT

Step 4: Full-scenario RL
  Do RL on more types of data (helpfulness, safety, etc.)
  So the model not only reasons well, but also converses naturally
```

**The most critical step is Step 2**: RL lets the model explore reasoning strategies on its own, rather than memorizing human reasoning processes.

In [ ]:
# Simulating the role of reward signals in RL training
print("=== RL Reasoning Training Simulation ===")
print()

print("Question: 15 + 27 = ?")
print()

# Simulate the model trying different reasoning paths
attempts = [
    ("15+20=35, 35+7=42", 42, True),
    ("15+27=42", 42, True),
    ("15+30=45, 45-3=42", 42, True),
    ("direct guess: 41", 41, False),
    ("10+20=30, 5+7=12, 30+12=42", 42, True),
]

for i, (reasoning, answer, correct) in enumerate(attempts):
    reward = 1 if correct else -1
    # Extra reward: detailed reasoning steps
    steps = len(reasoning.split(','))
    detail_bonus = min(steps * 0.05, 0.2)
    total_reward = reward + detail_bonus

    status = 'correct' if correct else 'wrong'
    print(f"Attempt {i+1}: {status} {reasoning}")
    print(f"  answer={answer}, correct={correct}, base_reward={reward}, detail_bonus={detail_bonus:.2f}")
    print(f"  total_reward={total_reward:+.2f}")
    print()

print("RL reinforces high-reward reasoning patterns and suppresses low-reward ones.")
print("The model gradually learns: more detailed and correct reasoning -> higher reward.")

## 5. Training Your Own Thinking Model

You do not need to start from scratch. Fine-tune an open-source model. Full process:

```
1. Choose a base model
   -> Qwen2.5-7B / DeepSeek-V3 / Llama-3 etc.
   -> Requirement: the base model itself must have decent reasoning ability

2. Prepare cold-start data (~5000 examples)
   -> Use a strong model (GPT-4/DeepSeek-R1) to generate question->thinking->answer
   -> Or extract from GSM8K, MATH etc. datasets
   -> Format:
      User: 357 x 289 = ?
      <think>
     357x200=71400
     357x80=28560
     357x9=3213
     71400+28560+3213=103173
      </think>
     103173

3. Cold-start SFT (~1-2 hours, single A100)
   -> Use LLaMA-Factory / Axolotl etc.
   -> Teach the model the <think>/answer format

4. RL training (the core, ~1-3 days)
   -> Use verl / OpenRLHF etc.
   -> Reward function: correct answer + correct format
   -> Math problems use rule-based verification (answer correctness is clear-cut)
   -> Code problems use test cases for verification

5. Rejection sampling + second round of SFT
   -> Use the trained model to generate more data
   -> Filter out wrong answers, keep correct ones
   -> Do another round of SFT to consolidate
```

**Cost estimate**: Based on Qwen2.5-7B, 4xA100, the full process takes about 3-5 days, costing a few thousand dollars.

In [ ]:
# Simulating cold-start data format
print("=== Cold-Start Data Format Examples ===")
print()

training_examples = [
    {
        "question": "A rectangle has length 12cm and width 8cm. Find its area.",
        "thinking": "Rectangle area = length x width\nArea = 12 x 8 = 96\nSo the area is 96 sq cm.",
        "answer": "96 sq cm"
    },
    {
        "question": "Xiaoming has 15 apples. He eats 3, then buys 7 more. How many does he have now?",
        "thinking": "Start: 15\nEats 3: 15 - 3 = 12\nBuys 7 more: 12 + 7 = 19\nSo he has 19 apples.",
        "answer": "19"
    },
    {
        "question": "357 x 289 = ?",
        "thinking": "357 x 200 = 71400\n357 x 80 = 28560\n357 x 9 = 3213\n71400 + 28560 + 3213 = 103173",
        "answer": "103173"
    }
]

for i, ex in enumerate(training_examples):
    print(f"--- Sample {i+1} ---")
    print(f"User:\n{ex['question']}\n")
    print(f"<think>\n{ex['thinking']}\n</think>\n{ex['answer']}")
    print()

print("This is the format the model learns during SFT.")
print("During RL, the model explores better thinking content on its own.")

## 6. The Aha Moment of Thinking Models

The DeepSeek-R1 paper mentions an interesting phenomenon: during RL training, the model spontaneously learned to reflect.

```
Model thinking process:
  Step 1: I think the answer is 42...
  Step 2: Wait, let me recalculate and check...
  Step 3: Oh wrong, 15+27 should be 42, but my reasoning just now had a problem...
  Step 4: Recalculate: 15+20=35, 35+7=42. Confirmed, the answer is 42.
```

**This was not taught by humans!** It is a strategy the model discovered on its own through RL -- the model found that "checking my own answer" improves accuracy, thereby earning more reward.

This is the magic of RL: instead of prescribing how to think, you only tell it "correct thinking gets reward," and the model explores the optimal thinking strategy on its own.

In [ ]:
# Simulating the emergence of reflection behavior in RL training
print("=== Emergence of Reflection Behavior in RL Training ===")
print()

def evaluate_reasoning(reasoning, correct_answer):
    """Evaluate reasoning process, returns (answer, is_correct, reward)"""
    import re
    numbers = re.findall(r'[\d.]+', reasoning)
    answer = float(numbers[-1]) if numbers else None
    correct = (answer == correct_answer)

    reward = 1.0 if correct else -1.0
    steps = reasoning.count('+') + reasoning.count('-') + reasoning.count('x') + reasoning.count('=')
    reward += min(steps * 0.05, 0.2)
    has_check = any(w in reasoning for w in ['verify', 'check', 'confirm', 'recalculate', 'wrong'])
    if has_check and correct:
        reward += 0.15
    return answer, correct, reward

correct_answer = 42.0
stages = [
    ("Early training", [
        "15+27=41",
        "15+27=44",
        "15+27=39",
    ]),
    ("Mid training (reflection begins)", [
        "15+27=41... wrong. Recalculate: 15+20=35, 35+7=42. Verify: 42-27=15",
        "15+20=35, 35+7=42",
        "10+20=30, 5+7=12, 30+12=42. Confirmed correct.",
    ]),
    ("Late training (stable reasoning)", [
        "15+20=35, 35+7=42. Verify: 42-27=15",
        "First compute 15+20=35, then add 7=42. Check: 42-15=27",
        "15+27: split into 15+20=35, 35+7=42. Verify 42-27=15, correct.",
    ]),
]

for stage_name, outputs in stages:
    print(f"{stage_name}:")
    total_reward = 0
    for reasoning in outputs:
        answer, correct, reward = evaluate_reasoning(reasoning, correct_answer)
        total_reward += reward
        status = 'correct' if correct else 'wrong'
        print(f"  {status} {reasoning[:60]}...")
        print(f"     answer={answer}, reward={reward:+.2f}")
    avg_reward = total_reward / len(outputs)
    print(f"  Average reward: {avg_reward:+.2f}")
    print()

print("Trend: early guessing -> mid-stage occasional reflection (high reward) -> late-stage reflection becomes habitual")
print("This is emergent behavior: no one taught the model to reflect; the RL reward made it discover this on its own.")

## 7. Limitations of Thinking Models

| Limitation | Description |
|:---|:---|
| **Slow** | The thinking process can be long (hundreds to thousands of tokens), requiring the user to wait |
| **Expensive** | Thinking tokens also cost money (APIs charge by token) |
| **Overthinking** | Even simple questions get lengthy reasoning -- "1+1=?" might trigger 500 words of reasoning |
| **Language mixing** | Thinking may mix languages, hurting readability |
| **Uncontrollable** | RL-trained thinking strategies are a black box, not necessarily matching human expectations |

**Best suited for**: math, programming, logical reasoning, scientific problems.
**Not suited for**: simple conversation, creative writing, emotional companionship.

## 8. Function Calling and Tool Use

CoT and Thinking make the model better at reasoning. But some problems cannot be solved by reasoning alone -- the model does not know today's weather, cannot execute code, and cannot directly query a database.

The approach of Function Calling is: **have the model output structured "call requests," which external programs execute and feed results back to the model**. The model itself does not execute anything; it only generates JSON describing "what function I want to call, with what parameters".

In [ ]:
# Function Calling complete workflow demo

# Step 1: Define available tools
tools = [
    {
        "name": "get_weather",
        "description": "Get weather information for a specified city",
        "parameters": {
            "city": {"type": "string", "description": "City name"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
        }
    },
    {
        "name": "calculate",
        "description": "Execute a math calculation",
        "parameters": {
            "expression": {"type": "string", "description": "Math expression, e.g. '2+3*4'"}
        }
    }
]

print("=== Defined tools ===")
for tool in tools:
    print(f"  {tool['name']}: {tool['description']}")
    print(f"    Parameters: {list(tool['parameters'].keys())}")
print()

In [ ]:
# Step 2: Simulate the model Function Calling output
# In practice, after fine-tuning, the model outputs structured JSON, not natural language

import json

# User question
user_query = "What is the temperature in Beijing today? Also calculate 357 * 289"

# The model "decides" to call two functions (this is the model's JSON output)
model_output = json.dumps({
    "tool_calls": [
        {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}},
        {"name": "calculate", "arguments": {"expression": "357*289"}}
    ]
}, ensure_ascii=False, indent=2)

print(f"User: {user_query}")
print(f"\nModel output (not natural language, but JSON):")
print(model_output)

In [ ]:
# Step 3: External program executes function calls and feeds results back

import random
import json

random.seed(42)

# Simulate tool execution
def execute_tool(name, arguments):
    if name == "get_weather":
        return {"temperature": 22, "condition": "sunny", "humidity": 45}
    elif name == "calculate":
        return {"result": eval(arguments["expression"])}
    return {"error": "unknown tool"}

# Execute and collect results
tool_calls = json.loads(model_output)["tool_calls"]
results = []
for call in tool_calls:
    result = execute_tool(call["name"], call["arguments"])
    results.append({"name": call["name"], "result": result})
    print(f"Called {call['name']}({call['arguments']})")
    print(f"  -> Returned: {result}")

print(f"\nThese results will be appended to the conversation history, then the model generates a final natural language response")
print(f'\nExample final reply: "Beijing is 22 degrees Celsius today, sunny. 357x289 = 103,173."')

### The Essence of Function Calling

From a technical perspective, Function Calling is **structured output (JSON mode) + prompt guidance**:

1. Tool descriptions (name, parameters, description) are concatenated into the system prompt
2. The model, after fine-tuning, learns to output JSON-formatted call requests instead of natural language
3. An external program parses the JSON, executes the function, and appends the result to the conversation
4. After seeing the result, the model generates a final natural language response

```
User question -> [model outputs JSON] -> external execution -> result appended to conversation -> [model outputs natural language]
```

Practical considerations:

| Consideration | Description |
|:---|:---|
| Tool descriptions must be clear | The model decides whether to call based on description; unclear descriptions lead to wrong calls |
| Parameter types must be strict | Use JSON Schema to constrain types; the model may generate invalid parameters |
| Parallel calls | One question may need multiple tool calls (as demonstrated above) |
| Error handling | Tool execution may fail; the model needs to handle error results |
| Do not use for simple computation | Let the model compute simple math itself; tools are for scenarios requiring external data |

## 9. Comparison of Mainstream Thinking Models

| Model | Training Method | Key Features |
|:---|:---|:---|
| **OpenAI o1** | Not public (suspected RL + CoT) | Strongest reasoning, closed-source |
| **DeepSeek-R1** | Cold-start SFT -> RL -> Rejection sampling -> SFT -> Full-scenario RL | Strongest open-source, detailed paper |
| **Qwen3** | Similar pipeline to R1 | Supports thinking/non-thinking dual mode |
| **Kimi K2** | Not public | Long context + reasoning |

**Common thread**: all use RL to train reasoning ability, and all distinguish between "thinking" and "answering" phases.

## 10. Practice: Enabling and Switching Thinking Modes

Each platform enables thinking differently, and these APIs may change with model versions. What follows is the most reliable usage based on currently available documentation. Remember to "check the model docs" rather than memorizing any single parameter as permanent.

---

#### 10.1 OpenAI reasoning models (o-series, GPT-5-series)

OpenAI reasoning models typically control thinking intensity via `reasoning_effort` or the `reasoning` configuration in the Responses API:

```text
Common values: "low" | "medium" | "high"
Some newer models also support "minimal" or "none". Refer to the official model docs.
```

**Key points**:
1. ✅ Models that support reasoning are OpenAI reasoning models, such as o3, o4-mini, GPT-5-series, etc.
2. ✅ Regular models like GPT-4o should not be treated as reasoning models by default.
3. ✅ The API typically does not return the full hidden thinking process, but may return reasoning token usage.
4. ✅ Billing and context usage must account for reasoning tokens.

Reference: [OpenAI reasoning guide](https://platform.openai.com/docs/guides/reasoning), [OpenAI models](https://platform.openai.com/docs/models).

---

#### 10.2 DeepSeek-R1 / DeepSeek thinking mode

Open-source thinking models like DeepSeek-R1 commonly use a **Chat Template** to separate the thinking section from the final answer. Some frontends collapse the model's `<think>...</think>` section:

```text
Model output format:
   <think>
  Analyze the problem conditions...
  Check if the calculation is correct...
   </think>
  42

Frontend handling:
5. ✅ By default, collapse <think>...</think> content
6. ✅ Only display the final answer after </think>
```

**Common misconception**: You cannot simply say "DeepSeek can never turn off thinking." Local R1 / R1-distill models will naturally output thinking format; but DeepSeek's official API now supports enabling or disabling thinking via parameters.

Common scenarios:
7. ✅ `deepseek-reasoner` returns `reasoning_content` and the final `content`.
8. ✅ Newer DeepSeek Chat/Reasoner APIs support switches like `thinking.enabled`; refer to the official docs for specifics.

Reference: [DeepSeek reasoning model](https://api-docs.deepseek.com/guides/reasoning_model), [DeepSeek thinking mode](https://api-docs.deepseek.com/guides/thinking_mode).

---

#### 10.3 Qwen3 (explicitly supports Thinking / Non-Thinking dual mode)

Qwen3 is one of the few models that explicitly supports thinking/non-thinking switching in its official chat template:

```text
Method 1: via chat template parameter
  enable_thinking=True   -> enter thinking mode
  enable_thinking=False  -> disable thinking, answer directly

Method 2: via commands in conversation
  user: /think       -> enable thinking
  user: /no_think    -> disable thinking
```

**How it works**: Qwen3 was trained on both thinking and non-thinking data, and switches behavior based on the `enable_thinking` toggle.

**Practical usage** (HuggingFace Transformers):
```python
# enable thinking
messages = [{"role": "user", "content": "357x289=?"}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False,
    enable_thinking=True
)

# disable thinking
text = tokenizer.apply_chat_template(
    messages, tokenize=False,
    enable_thinking=False
)
```

Reference: [Qwen3-8B model card](https://huggingface.co/Qwen/Qwen3-8B), [Qwen quickstart](https://qwen.readthedocs.io/en/v3.0/getting_started/quickstart.html).

---

#### 10.4 Anthropic Claude (Extended Thinking)

Claude's Extended Thinking also has version differences. Claude 3.7/4.x used to support manually setting `budget_tokens`, but newer Opus 4.7/4.8 documentation recommends adaptive thinking and no longer supports manual `budget_tokens`. So check the model version before writing code.

The old-style manual budget form looks like this:

```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=20000,
    thinking={
        "type": "enabled",
        "budget_tokens": 4000
    },
    messages=[{"role": "user", "content": "Prove that the square root of 2 is irrational"}]
)
```

Newer models may use `effort` or adaptive thinking. Reference: [Anthropic extended thinking docs](https://platform.claude.com/docs/en/build-with-claude/extended-thinking).

---

#### 10.5 Local inference (vLLM / SGLang / Ollama)

If deploying vLLM or SGLang with R1 distill models, thinking behavior is typically controlled by **chat template** and the inference framework's parser:

```bash
# vLLM with DeepSeek-R1-Distill-Qwen-7B
vllm serve deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
  --enable-reasoning \
  --reasoning-parser deepseek_r1
```

The frontend can choose to display, collapse, or filter `<think>...</think>`. This is not about the model "not thinking," but about the presentation layer deciding whether to show thinking content to the user.

---

#### Summary: Enabling Thinking across Platforms

| Platform/model | How to enable | Can disable? | Thinking content visible? |
|:---|:---|:---|:---|
| OpenAI reasoning models | `reasoning_effort` / `reasoning` config | Adjustable intensity; values vary by model | Usually not visible; only token usage shown |
| DeepSeek API | `deepseek-reasoner` or `thinking.enabled` | Newer API configurable; local R1 depends on template | API can return `reasoning_content` |
| Qwen3 | `enable_thinking=True/False`, `/think`, `/no_think` | Switchable | Template parser controls visibility |
| Claude | Extended Thinking / adaptive thinking | Depends on model version | Returned in separate blocks or per-version policy |
| Local R1 distill | Chat template + parser | Model behavior mostly fixed; frontend can filter | Can display or collapse |

In [ ]:
# ============================================================
# Practice: API calling examples for each platform (real runnable code)
# Note: requires environment variable API keys for actual calls
# If no key is set, prints equivalent curl command and skips
# ============================================================

import os

# ----------------------------------------------------------
# 1. OpenAI o3 API call
# ----------------------------------------------------------
openai_key = os.environ.get("OPENAI_API_KEY")
if openai_key:
    from openai import OpenAI
    client = OpenAI(api_key=openai_key)

    response = client.chat.completions.create(
        model="o3-mini",
        reasoning_effort="medium",  # low | medium | high
        messages=[
            {"role": "user", "content": "357 x 289 = ?"}
        ]
    )

    print("=== OpenAI o3-mini Result ===")
    print(f"Answer: {response.choices[0].message.content}")
    print(f"Token usage: {response.usage}")
else:
    print("=== OpenAI o3 API (OPENAI_API_KEY not set, skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.openai.com/v1/chat/completions \\')
    print('  -H "Content-Type: application/json" \\')
    print('  -H "Authorization: Bearer $OPENAI_API_KEY" \\')
    print('  -d \'{"model":"o3-mini","reasoning_effort":"medium","messages":[{"role":"user","content":"357x289=?"}]} \'')

# ----------------------------------------------------------
# 2. DeepSeek-R1 API call
# ----------------------------------------------------------
deepseek_key = os.environ.get("DEEPSEEK_API_KEY")
if deepseek_key:
    from openai import OpenAI
    client = OpenAI(
        api_key=deepseek_key,
        base_url="https://api.deepseek.com"
    )

    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[
            {"role": "user", "content": "357 x 289 = ?"}
        ]
    )

    print("\n=== DeepSeek-R1 Result ===")
    rc = response.choices[0].message.reasoning_content
    print(f"Thinking process: {rc[:200]}...")
    print(f"Final answer: {response.choices[0].message.content}")
else:
    print("\n=== DeepSeek-R1 API (DEEPSEEK_API_KEY not set, skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.deepseek.com/chat/completions \\')
    print('  -H "Content-Type: application/json" \\')
    print('  -H "Authorization: Bearer $DEEPSEEK_API_KEY" \\')
    print('  -d \'{"model":"deepseek-reasoner","messages":[{"role":"user","content":"357x289=?"}]} \'')

# ----------------------------------------------------------
# 3. Qwen3 local inference (HuggingFace Transformers + thinking toggle)
# ----------------------------------------------------------
QWEN3_MODEL = "Qwen/Qwen3-8B"
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    print("\n=== Qwen3 Local Inference ===")
    tokenizer = AutoTokenizer.from_pretrained(QWEN3_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        QWEN3_MODEL,
        torch_dtype="auto",
        device_map="auto"
    )

    messages = [{"role": "user", "content": "357 x 289 = ?"}]

    # --- Method A: enable thinking ---
    print("\n>>> enable_thinking=True (thinking mode enabled)")
    text_on = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=True
    )
    inputs = tokenizer(text_on, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024)
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"Output (with thinking tags): {result[:300]}...")

    # --- Method B: disable thinking ---
    print("\n>>> enable_thinking=False (thinking disabled, answer directly)")
    text_off = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False
    )
    inputs = tokenizer(text_off, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512)
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"Output: {result[:300]}...")

except Exception as e:
    print(f"\n=== Qwen3 Local Inference (skipped: {type(e).__name__}) ===")
    print(f"Reason: {e}")
    print()
    print("To run Qwen3 locally, execute:")
    print("  pip install transformers torch accelerate")
    print(f'  model = AutoModelForCausalLM.from_pretrained("{QWEN3_MODEL}", device_map="auto")')
    print("  tokenizer.apply_chat_template(messages, enable_thinking=True)")

# ----------------------------------------------------------
# 4. Claude Extended Thinking API call
# ----------------------------------------------------------
anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
if anthropic_key:
    import anthropic
    client = anthropic.Anthropic(api_key=anthropic_key)

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=20000,
        thinking={
            "type": "enabled",
            "budget_tokens": 4000
        },
        messages=[
            {"role": "user", "content": "Prove that the square root of 2 is irrational"}
        ]
    )

    print("\n=== Claude Extended Thinking Result ===")
    for block in response.content:
        if block.type == "thinking":
            print(f"Thinking process: {block.thinking[:200]}...")
        elif block.type == "text":
            print(f"Final answer: {block.text[:200]}...")
else:
    print("\n=== Claude API (ANTHROPIC_API_KEY not set, skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.anthropic.com/v1/messages \\')
    print('  -H "x-api-key: $ANTHROPIC_API_KEY" \\')
    print('  -H "anthropic-version: 2023-06-01" \\')
    print('  -H "content-type: application/json" \\')
    print('  -d \'{"model":"claude-sonnet-4-20250514","max_tokens":20000,"thinking":{"type":"enabled","budget_tokens":4000},"messages":[{"role":"user","content":"Prove sqrt(2) is irrational"}]} \'')

# ----------------------------------------------------------
# 5. Via OpenRouter (OpenAI-compatible API, supports multiple models)
# ----------------------------------------------------------
openrouter_key = os.environ.get("OPENROUTER_API_KEY")
if openrouter_key:
    from openai import OpenAI
    client = OpenAI(
        api_key=openrouter_key,
        base_url="https://openrouter.ai/api/v1"
    )

    response = client.chat.completions.create(
        model="deepseek/deepseek-r1",
        messages=[
            {"role": "user", "content": "What is entropy? Explain in one sentence."}
        ]
    )

    print("\n=== OpenRouter (DeepSeek-R1) Result ===")
    msg = response.choices[0].message
    reasoning = getattr(msg, "reasoning", None)
    if reasoning:
        print(f"Thinking process: {reasoning[:200]}...")
    print(f"Final answer: {msg.content[:200]}...")
else:
    print("\n=== OpenRouter API (OPENROUTER_API_KEY not set, skipping) ===")
    print("Register for a key: https://openrouter.ai/keys")
    print("Supported models: deepseek/deepseek-r1, openai/o1, anthropic/claude-sonnet-4 etc.")
    print()
    print("Equivalent curl:")
    print('curl https://openrouter.ai/api/v1/chat/completions \\')
    print('  -H "Authorization: Bearer $OPENROUTER_API_KEY" \\')
    print('  -H "Content-Type: application/json" \\')
    print('  -d \'{"model":"deepseek/deepseek-r1","messages":[{"role":"user","content":"What is entropy?"}]} ' "")

## 11. Hands-on: Training a Thinking Model

Below is a **teaching-version full pipeline**, based on Qwen2.5-7B. The goal is to understand the "think before answering" reasoning model. It is suitable for small-scale experiments; training a model with capabilities close to DeepSeek-R1/Qwen3 level requires much more data, RL sampling, reward design, and compute.

---

#### Step 1: Environment Setup

```bash
# Create virtual environment
conda create -n thinking-train python=3.10 -y
conda activate thinking-train

# Install core dependencies
pip install torch==2.4.0 transformers datasets accelerate
pip install vllm  # For efficient inference and generating training data

# Install training framework (choose one)
pip install llama-factory  # Recommended for beginners, has Web UI
# or
pip install axolotl        # More flexible, for advanced users
```

---

#### Step 2: Prepare Cold-start Data (~3000-5000 examples)

The core of cold-start data is the **question -> thinking process -> answer** triple.

**Data sources**:
1. Use a strong model (DeepSeek-R1 / GPT-4o) to generate
2. Extract from public datasets: GSM8K (elementary math), MATH (competition math), APPS (programming)
3. Mix Chinese and English data so the model does not bias toward one language

**Data format** (Qwen3 / DeepSeek style):
```json
[
  {
    "messages": [
      {"role": "user", "content": "A water tank: inlet pipe fills in 3 hours, outlet pipe drains in 5 hours. If both are open, how long to fill?"},
      {"role": "assistant", "content": "<think>\nInflow rate = 1/3 tank/hour\nOutflow rate = 1/5 tank/hour\nNet rate = 1/3 - 1/5 = 2/15 tank/hour\nTime to fill = 1/(2/15) = 15/2 = 7.5 hours\n</think>\n7.5 hours"}
    ]
  }
]
```

**Important**: thinking should be detailed but not excessive. For simple problems, write 50-100 words; for hard problems, 200-500 words. If every "1+1=?" gets 500 words of reasoning, the trained model will have a severe overthinking problem.

---

#### Step 3: Cold-start SFT (Teach the Model the Format)

This is the critical step that teaches the model the `<think>` / answer format.

**Using LLaMA-Factory** (recommended, has GUI so fewer mistakes):

```bash
# Start LLaMA-Factory Web UI
llamafactory-cli webui

# In the Web UI:
# 1. Select model: Qwen/Qwen2.5-7B-Instruct
# 2. Upload your cold-start dataset
# 3. Training type: Supervised Fine-Tuning
# 4. LoRA: rank=64, alpha=128
# 5. Learning rate: 5e-5, epochs: 3
# 6. Sequence length: 4096
# 7. batch_size: 4, gradient_accumulation: 4
# 8. Start training!
```

**Using command line**:

```bash
llamafactory-cli train \
    --model_name_or_path Qwen/Qwen2.5-7B-Instruct \
    --dataset my_cot_coldstart \
    --template qwen \
    --finetuning_type lora \
    --lora_rank 64 \
    --lora_alpha 128 \
    --output_dir ./output/qwen-sft-cot \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --lr_scheduler_type cosine \
    --logging_steps 10 \
    --save_steps 500 \
    --learning_rate 5e-5 \
    --num_train_epochs 3 \
    --bf16
```

**Expected result**: after training, the model will write thinking for every question, but quality depends on the cold-start data quality.

---

#### Step 4: RL Reasoning Training (the Core -- Making the Model Smarter)

SFT only makes the model "know the format"; RL is the key to making it "learn reasoning."

**Using verl** (recommended, same framework as DeepSeek-R1):

```bash
# Install verl
git clone https://github.com/volcengine/verl.git
cd verl
pip install -e .

# Prepare reward function dataset: only need (question, correct_answer)
# Format: {"prompt": "...", "answer": "..."}
```

**verl config file** (`r1_train.yaml`):

```yaml
# Model config
actor_rollout_ref:
  model:
    path: ./output/qwen-sft-cot  # Model after cold-start
    use_fused_kernels: true
  rollout:
    n: 4  # Generate 4 candidate answers per prompt
    temperature: 1.0

# Reward functions
reward_fn:
  # Math problems: compare model output with standard answer
  - name: math_verify
    weight: 0.7
  # Format check: ensure thinking tags are present
  - name: format_check
    weight: 0.2
  # Language consistency
  - name: language_consistency
    weight: 0.1

# Training config
trainer:
  n_gpus_per_node: 4
  nnodes: 1
  total_epochs: 1
  project_name: "my-thinking-model"
```

```bash
# Start RL training
python -m verl.trainer.main_ppo \
    --config-name r1_train.yaml
```

**Using OpenRLHF** (alternative, easier to get started):

```bash
# Install
git clone https://github.com/OpenRLHF/OpenRLHF.git
cd OpenRLHF
pip install -e .

# Ray cluster mode (4 GPUs)
ray start --head --num-gpus=4

python -m openrlhf.cli.train_ppo_ray \
    --ref_num_nodes 1 \
    --ref_num_gpus_per_node 2 \
    --reward_num_nodes 1 \
    --reward_num_gpus_per_node 1 \
    --actor_num_nodes 1 \
    --actor_num_gpus_per_node 1 \
    --pretrain ./output/qwen-sft-cot \
    --reward_fn math_verify \
    --save_path ./output/qwen-rl \
    --prompt_data ./data/math_prompts.jsonl
```

**Key tips for RL training**:

| Tip | Description |
|:---|:---|
| **Math problems are the best RL starting point** | Answer correctness is clear-cut, no manual reward annotation needed |
| **Format reward should not be too high** | Too high makes the model focus only on format, not correctness |
| **High temperature (0.7-1.0)** | RL needs exploration; low temperature suppresses discovery of new strategies |
| **Generate 4-8 candidates per prompt** | Sufficient sampling is needed to discover good reasoning paths |
| **Reward function is your "syllabus"** | What you reward, the model learns |

---

#### Step 5: Rejection Sampling + Second Round of SFT

```bash
# Use the trained model to generate answers for 5000 new questions
# Only keep samples with correct answers ("reject" wrong ones)
python scripts/rejection_sampling.py \
    --model ./output/qwen-rl \
    --prompts ./data/new_prompts.jsonl \
    --output ./data/rl_filtered.jsonl \
    --num_samples 4 \
    --keep_correct_only

# Use filtered high-quality data for another round of SFT
llamafactory-cli train \
    --model_name_or_path ./output/qwen-rl \
    --dataset rl_filtered \
    --template qwen \
    --finetuning_type lora \
    --output_dir ./output/qwen-final \
    --learning_rate 1e-5 \
    --num_train_epochs 2
```

---

#### Cost and Time Estimates

| Step | Hardware | Time | Cost (cloud GPU) |
|:---|:---|:---|:---|
| Cold-start SFT | 1xA100 (80G) | Hours | Depends on cloud provider pricing |
| RL reasoning training | 4xA100 (80G) minimum | Possibly 1 day to multiple days | Strongly depends on rollout count, problem count, and sample length |
| Rejection sampling + SFT | 1xA100 (80G) | Hours | Strongly depends on number of generated samples |

> Do not memorize a fixed price here. Small-scale math problem experiments might cost a few hundred dollars to complete the pipeline; building a stable, generalizable thinking model costs significantly more. To estimate real costs, first determine the number of problems, samples per problem, average generation length, training epochs, and GPU unit price.

---

#### Common Pitfalls for Beginners

1. **Cold-start data has thinking too short** -> model learns format but not reasoning -> fix: ensure thinking is at least 50+ words
2. **Reward function only rewards correctness** -> model learns to cheat (e.g., outputs answer directly without thinking) -> fix: add format reward
3. **Math dataset too easy** -> RL converges quickly but reasoning ability does not improve -> fix: mix at least 30% hard problems
4. **Training set and test set overlap** -> model memorizes rather than learning to reason -> fix: strictly split by problem ID into train/test
5. **Wrong chat template** -> thinking tags render incorrectly, model behavior is abnormal -> fix: verify against the official model documentation template

---

#### Easy Route: Use R1 Distill Models

If you only want to **use** a thinking model without training, directly use the distill models released by DeepSeek:

```python
# DeepSeek-R1 distill versions (already have thinking ability, no training needed)
# 1.5B version: suitable for toy projects
# 7B version:   suitable for research and experiments
# 14B version:  reasoning ability noticeably improved
# 32B version:  close to original R1 reasoning level
# 70B version:  strongest distill version

# Download and use directly
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# This model already knows how to think! Format is the same as R1.
# If you want to "disable thinking," you can train on non-thinking data based on this.
```

Distill models are essentially "student models" of R1 -- trained using R1's thinking data. Their reasoning ability is far stronger than regular models of the same size, but not as strong as the original R1.

In [ ]:
# ============================================================
# Practice: Training scripts (all functions are complete runnable implementations)
# ============================================================

import json
import os

import re

# ----------------------------------------------------------
# A. Cold-start data generation (using DeepSeek API to generate training data)
# ----------------------------------------------------------
def generate_coldstart_data(questions, output_path="coldstart_data.json"):
    """Use DeepSeek-R1 API to batch-generate question-thinking-answer training data"""
    deepseek_key = os.environ.get("DEEPSEEK_API_KEY")
    if not deepseek_key:
        print("SKIP: DEEPSEEK_API_KEY environment variable required")
        return []

    from openai import OpenAI
    client = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com")

    SYSTEM_PROMPT = """You are a math reasoning assistant. For each question:
1. Write out a detailed step-by-step reasoning process
2. Explain the calculation logic at each step
3. Give a concise final answer

Important: the output format must be:
<think>
(reasoning process)
</think>
(final answer)
"""

    dataset = []
    for i, question in enumerate(questions):
        print(f"[{i+1}/{len(questions)}] {question[:50]}...")
        try:
            response = client.chat.completions.create(
                model="deepseek-reasoner",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question}
                ]
            )
            thinking = response.choices[0].message.reasoning_content or ""
            answer = response.choices[0].message.content or ""

            content = f"<think>\n{thinking}\n</think>\n{answer}"
            dataset.append({
                "messages": [
                    {"role": "user", "content": question},
                    {"role": "assistant", "content": content}
                ]
            })
        except Exception as e:
            print(f"  Failed: {e}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(dataset)} examples to {output_path}")
    return dataset

# Example question set
SAMPLE_QUESTIONS = [
    "A rectangle has length 12cm and width 8cm. Find its area.",
    "Xiaoming has 15 apples. He eats 3, then buys 7 more. How many does he have now?",
    "357 x 289 = ?",
    "A water tank: inlet pipe fills in 3 hours, outlet pipe drains in 5 hours. If both open, how long to fill?",
    "In the arithmetic sequence 3, 7, 11, ..., what is the 20th term?",
]

if os.environ.get("DEEPSEEK_API_KEY"):
    generate_coldstart_data(SAMPLE_QUESTIONS[:2])  # Demo 2 examples
else:
    print("=== generate_coldstart_data function defined ===")
    print("Set DEEPSEEK_API_KEY then call generate_coldstart_data(questions)")
    print("\nExample data structure:")
    example = {
        "messages": [
            {"role": "user", "content": "357 x 289 = ?"},
            {
                "role": "assistant",
                "content": (
                    "<think>\n"
                    "357 x 200 = 71400\n"
                    "357 x 80  = 28560\n"
                    "357 x 9   = 3213\n"
                    "71400 + 28560 + 3213 = 103173\n"
                    "</think>\n"
                    "103173"
                )
            }
        ]
    }
    print(json.dumps(example, ensure_ascii=False, indent=2))

# ----------------------------------------------------------
# B. RL Reward function (complete implementation, directly usable with verl/OpenRLHF)
# ----------------------------------------------------------
def extract_thinking_and_answer(completion):
    """Extract thinking content and final answer from model output"""
    thinking = ""
    answer = completion.strip()

    think_match = re.search(r"<think>(.*?)</think>", completion, re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        answer = completion[think_match.end():].strip()

    return thinking, answer

def normalize_number(text):
    """Extract and normalize numbers from text, used for answer comparison"""
    text = text.strip().replace(" ", "").replace(",", "")
    match = re.search(r"-?[\d.]+(?:/-?[\d.]+)?", text)
    if match:
        num_str = match.group()
        if "/" in num_str:
            parts = num_str.split("/")
            try:
                return str(float(parts[0]) / float(parts[1]))
            except (ValueError, ZeroDivisionError):
                return num_str
        try:
            return str(float(num_str))
        except ValueError:
            return num_str
    return text

def compute_reward(completion, ground_truth):
    """
    Compute reward for a model output.
    Reward components: correct format +0.15 | correct answer +1.0 | thinking length +0.05
    Penalties: severely wrong format -1.0 | wrong answer -0.5
    """
    reward = 0.0
    thinking, answer = extract_thinking_and_answer(completion)

    has_thinking = "<think>" in completion
    has_answer_tag = "</think>" in completion

    if not has_answer_tag:
        return -1.0

    if has_thinking and has_answer_tag:
        reward += 0.15

    pred_norm = normalize_number(answer)
    gt_norm = normalize_number(ground_truth)

    if pred_norm == gt_norm:
        reward += 1.0
    else:
        try:
            pv = float(pred_norm)
            gv = float(gt_norm)
            if abs(pv - gv) / max(abs(gv), 1e-8) < 0.001:
                reward += 1.0
            else:
                reward -= 0.5
        except ValueError:
            reward -= 0.5

    if len(thinking) >= 30:
        reward += 0.05

    return reward

# Test reward function
print("=== RL Reward Function Test ===\n")
test_cases = [
    (
        "<think>15+20=35, 35+7=42</think>\n42",
        "42",
    ),
    (
        "<think>15+27=41</think>\n41",
        "42",
    ),
    (
        "No thinking process\nanswer is 42",
        "42",
    ),
    (
        "<think>calculating...</think>\n7.5",
        "7.5",
    ),
]

for completion, gt in test_cases:
    reward = compute_reward(completion, gt)
    print(f"Reward: {reward:+.2f} | Output: {completion[:50]}...")
print()

# ----------------------------------------------------------
# C. Rejection sampling (complete implementation, directly usable for second-round SFT data preparation)
# ----------------------------------------------------------
def rejection_sampling(model, tokenizer, prompts, num_samples=4):
    """
    Generate num_samples candidate answers for each prompt,
    only keep those with correct answers, selecting the one with the most detailed thinking.

    Args:
        model: HuggingFace model instance
        tokenizer: HuggingFace tokenizer instance
        prompts: [{"question": "...", "answer": "..."}, ...]
        num_samples: how many candidates to generate per question
    Returns:
        filtered training data list
    """
    import torch
    device = next(model.parameters()).device
    filtered_data = []
    total_candidates = 0
    total_correct = 0

    for i, prompt in enumerate(prompts):
        question = prompt["question"]
        ground_truth = prompt["answer"]
        candidates = []

        for _ in range(num_samples):
            messages = [{"role": "user", "content": question}]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    temperature=0.8,
                    do_sample=True,
                )

            completion = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True
            )

            thinking, answer = extract_thinking_and_answer(completion)
            is_correct = compute_reward(completion, ground_truth) > 0.5
            candidates.append((thinking, answer, is_correct))
            total_candidates += 1
            if is_correct:
                total_correct += 1

        correct = [(t, a) for t, a, ok in candidates if ok]
        if correct:
            best_t, best_a = max(correct, key=lambda x: len(x[0]))
            content = f"<think>\n{best_t}\n</think>\n{best_a}"
            filtered_data.append({
                "messages": [
                    {"role": "user", "content": question},
                    {"role": "assistant", "content": content}
                ]
            })

        if (i + 1) % 10 == 0:
            print(f"Progress: {i+1}/{len(prompts)} | "
                  f"candidates={total_candidates} correct={total_correct} "
                  f"kept={len(filtered_data)}")

    return filtered_data

print("=== rejection_sampling function defined ===")
print()
print("Usage:")
print("  from transformers import AutoModelForCausalLM, AutoTokenizer")
print('  model = AutoModelForCausalLM.from_pretrained("./output/qwen-rl")')
print("  prompts = [{'question': '...', 'answer': '...'}, ...]")
print("  new_data = rejection_sampling(model, tokenizer, prompts, num_samples=4)")
print()

# ----------------------------------------------------------
# D. Recommended directory structure
# ----------------------------------------------------------
print("=== Recommended Directory Structure ===")
print(r"""
my-thinking-model/
+-- data/
|   +-- coldstart.jsonl       # Cold-start SFT data (~3000 examples)
|   +-- rl_prompts.jsonl      # RL training question set (~1000 examples)
|   +-- rl_filtered.jsonl     # Data after rejection sampling
+-- configs/
|   +-- sft_coldstart.yaml    # LLaMA-Factory config
|   +-- rl_verl.yaml          # verl config
+-- output/
|   +-- qwen-sft-cot/         # Model after cold-start
|   +-- qwen-rl/              # Model after RL training
|   +-- qwen-final/           # Final model
+-- scripts/
|   +-- gen_coldstart.py      # Generate cold-start data (using function above)
|   +-- rejection_sampling.py # Rejection sampling (using function above)
|   +-- eval_thinking.py      # Evaluation script
+-- README.md
""")
print()
print("All functions above are complete runnable implementations (not pseudocode).")
print("Core idea: math problem answers can be automatically verified, making them the best data type for RL starting point.")

## Summary

1. ✅  **CoT** = have the model write out its reasoning process, using intermediate results to assist subsequent reasoning
2. ✅  **Thinking model** = engineered packaging of CoT, with thinking possibly separated by special tokens or API blocks
3. ✅  **Training pipeline**: Cold-start SFT -> RL reasoning training -> Rejection sampling -> Full-scenario RL
4. ✅  **RL is the key**: instead of prescribing how to think, only reward "correct thinking"
5. ✅  **Reflection is emergent**: the model may spontaneously learn to check and correct itself during RL
6. ✅  **Each platform enables thinking differently**: OpenAI, DeepSeek, Qwen3, Claude all require checking the corresponding model version documentation
7. ✅  **Qwen3** is one of the few models that explicitly supports runtime thinking/non-thinking switching
8. ✅  **Training yourself**: the teaching-version pipeline can demonstrate the concept; real high-quality thinking model costs cannot be summarized with a fixed price
9. ✅  **Easy route**: directly use DeepSeek-R1 distill models or Qwen3 thinking model, ready to use out of the box
10. **Function Calling** = model outputs structured JSON describing "what function to call," external program executes and feeds results back to the model

**One-sentence summary**: Regular model = give answer directly; Thinking model = draft first, then answer. The drafting ability is not taught by humans, but shaped by RL rewards. Now you know how to check API switches, and why training a thinking model is not just about adding `<think>` tags.

## Exercises

**Exercise 1: Implement Self-Consistency voting**

Given 5 candidate answers to the same problem, write a function that extracts the final numeric answer and performs majority voting. Compare accuracy between "use the first answer" and "use majority vote."

**Exercise 2: Write a `<think>` format checker**

Implement a function that checks whether an output contains paired `<think>...</think>` tags and whether the final answer appears after `</think>`. Test it on 5 normal and abnormal examples.

**Exercise 3: Analyze reward hacking**

Construct examples with "long thinking but wrong answer," "short thinking but correct answer," and "correct format but empty content." Design a reward rule that rewards correctness and format while avoiding a pure preference for longer thinking.